# 01 — Data Inspection & SQLite Load

Inspects each source CSV, stacks the three beneficiary summary years, and writes all tables to `cms_data.db`.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
RAW = Path("../data/raw")
DB  = Path("../cms_data.db")

FILES = {
    "inpatient_claims":         RAW / "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv",
    "outpatient_claims":        RAW / "DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv",
    "prescription_drug_events": RAW / "DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv",
}

BENE_FILES = {
    2008: RAW / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    2009: RAW / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    2010: RAW / "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv",
}

---
## 1. Inspect single-file tables

In [ ]:
def inspect(df: pd.DataFrame, name: str) -> None:
    print(f"{'='*60}")
    print(f"  {name}")
    print(f"  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    print(f"{'='*60}")

    print("\n--- dtypes ---")
    print(df.dtypes.to_string())

    print("\n--- sample (3 rows) ---")
    display(df.head(3))

    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print("\n--- null counts (non-zero columns only) ---")
        print(nulls.to_string())

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    cost_cols = [c for c in numeric_cols if any(k in c for k in ("AMT", "CST", "PAY", "REIMB", "RES"))]
    if cost_cols:
        print("\n--- cost / payment columns ---")
        display(df[cost_cols].describe().round(2))

In [ ]:
tables = {}

for name, path in FILES.items():
    df = pd.read_csv(path, low_memory=False)
    tables[name] = df
    inspect(df, name)

---
## 2. Beneficiary summary — stack 2008 / 2009 / 2010

In [ ]:
bene_frames = []
for year, path in BENE_FILES.items():
    df = pd.read_csv(path, low_memory=False)
    df.insert(0, "YEAR", year)
    bene_frames.append(df)
    print(f"{year}: {df.shape[0]:,} rows")

bene = pd.concat(bene_frames, ignore_index=True)
tables["beneficiary_summary"] = bene

print(f"\nCombined: {bene.shape[0]:,} rows × {bene.shape[1]} columns")

In [ ]:
inspect(bene, "beneficiary_summary")

In [ ]:
# Chronic condition flag distribution (1 = has condition, 2 = does not)
sp_cols = [c for c in bene.columns if c.startswith("SP_")]
print("Chronic condition prevalence (% of beneficiary-years with flag = 1):\n")
prevalence = (bene[sp_cols] == 1).mean().mul(100).round(1).sort_values(ascending=False)
print(prevalence.to_string())

---
## 3. Load to SQLite

In [ ]:
conn = sqlite3.connect(DB)

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)
    count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:<30} {count:>10,} rows  →  written")

conn.close()
print(f"\nDatabase: {DB.resolve()}")